# Evaluating Prompt Engineering Methods On ABSA Dev

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



In [ ]:
def generate_chat(model, tokenizer, user_content: str, max_new_tokens: int):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import json

In [ ]:
def extract_first_json(text:str):
    #this function aims to find {...} from the raw generated text
    if text is None:
        return None
    text = text.strip()
    return json.loads(text)

##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here 

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
        return rows

# Evaluating Prompt Engineering Methods On ABSA Dev

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)

In [ ]:
def get_allowed_categories(rows):
    allowed_categories = set()
    for r in rows:
        for op in r["labels"]:
            allowed_categories.add(op["category"])
    return allowed_categories

ALLOWED_CATEGORIES = get_allowed_categories(absa_train)

In [ ]:
def pick_absa_shots(rows, shots):
    if shots!=3 and shots!=5:
        raise ValueError("shots must be 3 or 5")
    picked_shots = []
    for r in rows:
        picked_shots.append(r)
        if len(picked_shots)==shots:
            break
    return picked_shots

absa_three_shots = pick_absa_shots(absa_train, 3)
absa_five_shots = pick_absa_shots(absa_train, 5)

In [ ]:
def build_absa_prompt(text, allowed_categories, picked_shots=None):
    cats_str = ", ".join(allowed_categories)

    header = (
        "أنت نظام لاستخراج مشاعر الجوانب (Aspect-Based Sentiment) من مراجعات الفنادق باللغة العربية.\n\n"
        "المهمة:\n"
        "استخرج جميع الآراء (opinions) الموجودة في النص.\n"
        "كل رأي يجب أن يكون زوجاً من:\n"
        "- category: جانب محدد بصيغة مثل HOTEL#CLEANLINESS\n"
        "- polarity: واحدة من {positive, negative, neutral, conflict}\n\n"
        "قيود مهمة جداً:\n"
        f"- استخدم فقط الفئات التالية (categories): {cats_str}\n"
        "صيغة الإخراج بالضبط:\n"
        '{"labels":[{"category":"...","polarity":"..."}, ...]}\n'
        "لا تكتب ولا تكتب اي شيء آخر '''json فقط اخرج بصياغ المكتوب اعلى\n"
        "- إذا لم تجد أي آراء، أرجع: {\"labels\":[]}\n"
        "لا تخمّن. لا تضف أي رأي إلا إذا كان واضحًا ومذكورًا في النص."
        "أخرج بحد أقصى 8 آراء. إذا زادت، اختر الأوضح فقط."
    )

    examples = ""
    if picked_shots:
        lines = ["أمثلة (Examples):"]
        for shot in picked_shots:
            lines.append(f'النص: {shot["text"]}')
            lines.append("الإجابة: " + json.dumps({"labels": shot["labels"]}, ensure_ascii=False))
            lines.append("")
        examples = "\n".join(lines).strip() + "\n\n"

    return f"{header}\n{examples}النص:\n{text}\n"

In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}
def format_absa_prediction(obj, allowed_categories):
    if not isinstance(obj, dict) or "labels" not in obj:
        return None
    
    labels = obj["labels"]
    if not isinstance(labels, list):
        return None
    
    allowed_set = set(allowed_categories)
    out = set()
    for label in labels:
        if not isinstance(label, dict):
            continue
        cat = label["category"]
        pol = label["polarity"]
        if isinstance(cat, str): cat = cat.strip()
        if isinstance(pol, str): pol = pol.strip()

        if cat in allowed_set and pol in ABSA_POLARITIES:
            out.add((cat,pol))
    return out

In [ ]:
def f1_micro(y_true, y_pred):
    tp = fp = fn = 0
    for true, pred in zip(y_true, y_pred):
        tp += len(true&pred)
        fp += len(pred-true)
        fn += len(true-pred)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) else 0.0
    return precision, recall, f1

In [ ]:
from tqdm import tqdm

In [ ]:
def eval_absa_prompting(rows, allowed_categories, picked_shots = None):
    y_true, y_pred = [], []
    bad = 0
    
    for r in tqdm(rows):
        text = r["text"]
        prompt = build_absa_prompt(text, allowed_categories, picked_shots)
        print("\nText:", text)
        raw = generate_chat(model, tokenizer, prompt, max_new_tokens=1000)
        print(f"\nRaw: {raw}")
        block = extract_first_json(raw)
        pred = format_absa_prediction(block, allowed_categories)

        if pred is None:
            bad+= 1
            pred = set()
        
        true = set()
        for x in r["labels"]:
            true.add((x["category"], x["polarity"]))
        y_true.append(true)
        y_pred.append(pred)

        print("True:", true)
        print("Pred:", pred)
    
    p, rc, f1 = f1_micro(y_true, y_pred)
    return{"p":p, "rc":rc, "micro_f1":f1, "invalid_json":bad, "n":len(rows)}


In [ ]:
absa_dev_eval_zeroS = eval_absa_prompting(absa_dev, ALLOWED_CATEGORIES)

In [ ]:
print(absa_dev_eval_zeroS)

#### An analysis while viewing the debugging prints while it is generating that when a text has a many positives and one negative it just selects most categories it can which i limited it to 8 (I had a long issue with this because before it would print every category as positive and would get cut off from the tokens limit but fixed it with changin gthe prompt) and the polarity for all 8 will be positive and also even if there was only positive talk but many compliments on 2 categories it would do the same thing get 8 and positive all of them

In [ ]:
save_path_absa_dev_eval_zeroS = "drive/MyDrive/FYP/absa/prompt_engineering_results/absa_dev_eval_zeroS.json"
with open(save_path_absa_dev_eval_zeroS, "w", encoding="utf-8") as f:
    json.dump(absa_dev_eval_zeroS, f, ensure_ascii=False, indent=2)

In [ ]:
absa_dev_eval_threeS = eval_absa_prompting(absa_dev, ALLOWED_CATEGORIES, absa_three_shots)
absa_dev_eval_fiveS = eval_absa_prompting(absa_dev, ALLOWED_CATEGORIES, absa_five_shots)

In [ ]:
print(absa_dev_eval_threeS)
print(absa_dev_eval_fiveS)

In [ ]:
zeroS_path = "drive/MyDrive/FYP/absa/prompt_engineering_results/absa_dev_eval_zeroS.json"
with open(zeroS_path, "r", encoding="utf-8") as f:
    zeroS_results = json.load(f)
print(zeroS_results)

In [ ]:
final_results = {
    "zeroS": zeroS_results,
    "threeS": absa_dev_eval_threeS,
    "fiveS": absa_dev_eval_fiveS
}
final_results_save_path = "drive/MyDrive/FYP/absa/prompt_engineering_results/absa_dev_eval_final.json"
with open(final_results_save_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

## ANALYSIS
### Zero-Shot
####guessing too many aspects (low precision)
####missing many true ones (low recall)

### Three-Shot
#### demonstrations teach the model what not to predict
#### hallucinations are reduced

### Five-Shot
#### Adding more examples helps the model be more conservative, but does not significantly help it discover new aspects.

### Summmary
#### Prompting alone struggles with complex Arabic ABSA, achieving only moderate performance even with few-shot prompting
#### Few-shot prompting mainly improves precision rather than recall, indicating limited capacity for structured extraction.
#### Wont be adding more than 5 so the model does not overfit the examples

In [ ]:
absa_test_eval_zeroS = eval_absa_prompting(absa_test, ALLOWED_CATEGORIES)
absa_test_eval_threeS = eval_absa_prompting(absa_test, ALLOWED_CATEGORIES, absa_three_shots)
absa_test_eval_fiveS = eval_absa_prompting(absa_test, ALLOWED_CATEGORIES, absa_five_shots)

In [ ]:
print(absa_test_eval_zeroS)
print(absa_test_eval_threeS)
print(absa_test_eval_fiveS)

In [ ]:
final_test_results_path = "drive/MyDrive/FYP/absa/prompt_engineering_results/absa_test_eval_results.json"
final_test_results = {
    "zeroS": absa_test_eval_zeroS,
    "threeS": absa_test_eval_threeS,
    "fiveS": absa_test_eval_fiveS
}
with open(final_test_results_path, "w", encoding="utf-8") as f:
    json.dump(final_test_results, f, ensure_ascii=False, indent=2)